In [1]:
#Imports
import pandas as pd
from altair import Chart, X, Y, Color, Scale
import altair as alt
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#Load data U.S. Population by county 2020-2024 source: https://www.census.gov/data/tables/time-series/demo/popest/2020s-total-cities-and-towns.html
pop_df = pd.read_csv('https://www2.census.gov/programs-surveys/popest/datasets/2020-2024/cities/totals/sub-est2024.csv', encoding='latin-1')
pop_df.drop_duplicates()
print(pop_df.head())
print(pop_df.shape)

   SUMLEV  STATE  COUNTY  PLACE  COUSUB  CONCIT  PRIMGEO_FLAG FUNCSTAT  \
0      40      1       0      0       0       0             0        A   
1     162      1       0    124       0       0             0        A   
2     162      1       0    460       0       0             0        A   
3     162      1       0    484       0       0             0        A   
4     162      1       0    676       0       0             0        A   

              NAME   STNAME  ESTIMATESBASE2020  POPESTIMATE2020  \
0          Alabama  Alabama            5025369          5033094   
1   Abbeville city  Alabama               2349             2349   
2  Adamsville city  Alabama               4393             4381   
3     Addison town  Alabama                661              659   
4       Akron town  Alabama                229              228   

   POPESTIMATE2021  POPESTIMATE2022  POPESTIMATE2023  POPESTIMATE2024  
0          5049196          5076181          5117673          5157699  
1       

In [3]:
# Load Single Family Home Data From Zillow
hv_df = pd.read_csv('Data/Metro_zhvi_uc_sfr_tier_0.33_0.67_sm_sa_month.csv')
hv_df = hv_df.dropna().reset_index(drop=True)
print(hv_df.shape)
hv_df.head()

(303, 317)


,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31
0,394913,1,"New York, NY",msa,NY,216204.906949,217118.726894,218037.699710,219896.351318,221811.278678,...,702182.814240,704969.033163,707107.919763,708900.805287,710150.247552,710896.834350,712302.053289,714730.852730,718376.989056,722152.928657
1,753899,2,"Los Angeles, CA",msa,CA,229297.470433,230172.549865,231311.720436,233582.033926,236045.928058,...,999743.331098,995425.637330,990670.231609,986095.146144,983311.895154,981993.638461,983107.386702,985799.648759,989828.455467,994554.776693
2,394463,3,"Chicago, IL",msa,IL,155779.960126,155931.205194,156209.724698,156902.542597,157730.488497,...,343920.881014,344749.181516,345226.838090,345740.750099,346716.042282,347812.294196,349197.912152,350501.179573,352118.031490,354044.365771
3,394692,5,"Houston, TX",msa,TX,124201.785021,124222.275651,124132.587009,124187.678168,124237.524336,...,311422.028483,310507.749997,309567.114049,308577.371747,307742.508535,307038.679976,306588.438186,306205.991455,306032.499583,306061.376066
4,395209,6,"Washington, DC",msa,VA,190554.271674,190724.956272,191018.183957,191687.549705,192690.707732,...,613647.858539,613390.166609,612066.818598,610552.119406,609282.350791,608480.941311,608399.190942,608991.010218,610130.502925,611293.161618


In [4]:
# Add City and State Columns
copy_hv_df = hv_df.copy()
copy_hv_df[['City', 'State']] = hv_df['RegionName'].str.split(', ', expand=True)
copy_hv_df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,City,State
0,394913,1,"New York, NY",msa,NY,216204.906949,217118.726894,218037.699710,219896.351318,221811.278678,...,707107.919763,708900.805287,710150.247552,710896.834350,712302.053289,714730.852730,718376.989056,722152.928657,New York,NY
1,753899,2,"Los Angeles, CA",msa,CA,229297.470433,230172.549865,231311.720436,233582.033926,236045.928058,...,990670.231609,986095.146144,983311.895154,981993.638461,983107.386702,985799.648759,989828.455467,994554.776693,Los Angeles,CA
2,394463,3,"Chicago, IL",msa,IL,155779.960126,155931.205194,156209.724698,156902.542597,157730.488497,...,345226.838090,345740.750099,346716.042282,347812.294196,349197.912152,350501.179573,352118.031490,354044.365771,Chicago,IL
3,394692,5,"Houston, TX",msa,TX,124201.785021,124222.275651,124132.587009,124187.678168,124237.524336,...,309567.114049,308577.371747,307742.508535,307038.679976,306588.438186,306205.991455,306032.499583,306061.376066,Houston,TX
4,395209,6,"Washington, DC",msa,VA,190554.271674,190724.956272,191018.183957,191687.549705,192690.707732,...,612066.818598,610552.119406,609282.350791,608480.941311,608399.190942,608991.010218,610130.502925,611293.161618,Washington,DC


In [5]:
# Dictionary to map States to State Code
state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
    'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}
pop_df['State'] = pop_df['STNAME'].map(state_to_code)
pop_df = pop_df.rename(columns={'NAME': 'City'})
selected_columns = ['City', 'State', 'POPESTIMATE2020','POPESTIMATE2021','POPESTIMATE2022','POPESTIMATE2023', 'POPESTIMATE2024']
# Filter the dataframe
pop_df = pop_df[selected_columns]
# Rename name to City

print(pop_df.shape)
pop_df.head()

(81334, 7)


,City,State,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024
0,Alabama,AL,5033094,5049196,5076181,5117673,5157699
1,Abbeville city,AL,2349,2329,2328,2337,2370
2,Adamsville city,AL,4381,4315,4251,4204,4181
3,Addison town,AL,659,665,670,669,676
4,Akron town,AL,228,229,226,229,233


In [6]:
yearly_hv_df = copy_hv_df[["City", "State"]].copy()
#Add 2020 column
yearly_hv_df["2020"] = hv_df.filter(like="2020").mean(axis=1)
#Add 2021 column
yearly_hv_df["2021"] = hv_df.filter(like="2021").mean(axis=1)
#Add 2022 column
yearly_hv_df["2022"] = hv_df.filter(like="2022").mean(axis=1)
#Add 2023 column
yearly_hv_df["2023"] = hv_df.filter(like="2023").mean(axis=1)
#Add 2024 column
yearly_hv_df["2024"] = hv_df.filter(like="2024").mean(axis=1)
#Describe single family home value dataframe
print(yearly_hv_df.shape)
# yearly_hv_df[["2020", "2021", "2022", "2023", "2024"]].describe()
yearly_hv_df['City'] = yearly_hv_df['City'] + ' City'
yearly_hv_df.head()

(303, 7)


,City,State,2020,2021,2022,2023,2024
0,New York City,NY,507475.582817,562746.528727,614144.251278,634307.114784,679571.311568
1,Los Angeles City,CA,703264.213377,804769.886619,915313.837111,915145.855240,978978.214329
2,Chicago City,IL,248213.325786,276656.929200,303204.860699,312660.406888,332432.591997
3,Houston City,TX,231235.318154,261348.376471,306526.892394,309995.585013,312483.114267
4,Washington City,DC,470273.997269,526014.528938,567450.243446,574252.577488,598864.835301


In [7]:
yearly_hv_df['City'] = yearly_hv_df['City'].str.strip().str.upper()
yearly_hv_df['State'] = yearly_hv_df['State'].str.strip().str.upper()
pop_df['City'] = pop_df['City'].str.strip().str.upper()
pop_df['State'] = pop_df['State'].str.strip().str.upper()

In [8]:
pop_df = pop_df.drop_duplicates(subset=['City', 'State'])
print(pop_df.shape)
pop_df.head()

(43907, 7)


,City,State,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024
0,ALABAMA,AL,5033094,5049196,5076181,5117673,5157699
1,ABBEVILLE CITY,AL,2349,2329,2328,2337,2370
2,ADAMSVILLE CITY,AL,4381,4315,4251,4204,4181
3,ADDISON TOWN,AL,659,665,670,669,676
4,AKRON TOWN,AL,228,229,226,229,233


In [9]:
yearly_hv_df.head()

,City,State,2020,2021,2022,2023,2024
0,NEW YORK CITY,NY,507475.582817,562746.528727,614144.251278,634307.114784,679571.311568
1,LOS ANGELES CITY,CA,703264.213377,804769.886619,915313.837111,915145.855240,978978.214329
2,CHICAGO CITY,IL,248213.325786,276656.929200,303204.860699,312660.406888,332432.591997
3,HOUSTON CITY,TX,231235.318154,261348.376471,306526.892394,309995.585013,312483.114267
4,WASHINGTON CITY,DC,470273.997269,526014.528938,567450.243446,574252.577488,598864.835301


In [10]:
# Merge Does is not working as expected
pop_yearly_hv_df = pd.merge(yearly_hv_df, pop_df, on=['City','State'], how='left')
pop_yearly_hv_df = pop_yearly_hv_df.dropna()
print(pop_yearly_hv_df.shape)
pop_yearly_hv_df.head()

(278, 12)


,City,State,2020,2021,2022,2023,2024,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024
0,NEW YORK CITY,NY,507475.582817,562746.528727,614144.251278,634307.114784,679571.311568,8740306.0,8453772.0,8356179.0,8390888.0,8478072.0
1,LOS ANGELES CITY,CA,703264.213377,804769.886619,915313.837111,915145.855240,978978.214329,3896329.0,3830987.0,3832998.0,3847428.0,3878704.0
2,CHICAGO CITY,IL,248213.325786,276656.929200,303204.860699,312660.406888,332432.591997,2745196.0,2706324.0,2684076.0,2699144.0,2721308.0
3,HOUSTON CITY,TX,231235.318154,261348.376471,306526.892394,309995.585013,312483.114267,2298945.0,2291070.0,2314258.0,2346908.0,2390125.0
5,PHILADELPHIA CITY,PA,262609.376620,297219.246374,327023.807993,343360.538910,364650.172874,1600788.0,1589918.0,1570554.0,1563349.0,1573916.0


In [11]:
years = [2020, 2021, 2022, 2023, 2024]
rows = []
for idx, row in pop_yearly_hv_df.iterrows():
    for year in years:
        rows.append({
            'City': row['City'],
            'Year': year,
            'House_Value': row[f'{year}'],
            'Population': row[f'POPESTIMATE{year}']
        })
df_long = pd.DataFrame(rows)
df_long.head()

,City,Year,House_Value,Population
0,NEW YORK CITY,2020,507475.582817,8740306.0
1,NEW YORK CITY,2021,562746.528727,8453772.0
2,NEW YORK CITY,2022,614144.251278,8356179.0
3,NEW YORK CITY,2023,634307.114784,8390888.0
4,NEW YORK CITY,2024,679571.311568,8478072.0


In [14]:
scatter = alt.Chart(df_long).mark_circle(size=120).encode(
    x=alt.X('Population:Q', title='Population Count', scale=alt.Scale(zero=False)),
    y=alt.Y('House_Value:Q', title='House Value ($)', scale=alt.Scale(zero=False)),
    color=alt.Color('Year:O', title='Year', scale=alt.Scale(scheme='viridis')),
    tooltip=['City', 'Year', 'Population', 'House_Value']
).properties(
    width=200,
    height=200
).facet(
    facet='City:N',
    columns=4,
    title='Correlation: Population vs. House Value per City (2020-2024)'
).resolve_scale(
    x='independent', # Custom scale for each city's population range
    y='independent'  # Custom scale for each city's price range
)

scatter

alt.FacetChart(...)